In [28]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import jaccard_score

# Sample data loading
resumes = pd.read_csv("C:/Users/Acer/Desktop/ARSwithPredictiveAnalytics/Data-Training/notebooks/all_resumes_extractedentity.csv")  # Contains 'resume_id' and 'raw_extracted_from_resumes'
jobs = pd.read_csv("C:/Users/Acer/Desktop/ARSwithPredictiveAnalytics/Data-Training/notebooks/all_jobreq_extractedentity.csv")  # Contains 'job_id', 'job_title', and 'raw_extracted_from_job_req'

# Add auto-incrementing IDs if they don't exist
resumes.insert(0, 'resume_id', range(1, len(resumes) + 1))
jobs.insert(0, 'job_id', range(1, len(jobs) + 1))

# Label encoding job titles
label_encoder = LabelEncoder()
jobs['encoded_job_title'] = label_encoder.fit_transform(jobs['job_title'])

# Function to compute Jaccard Similarity
def compute_jaccard_similarity(text1, text2):
    set1, set2 = set(text1.split()), set(text2.split())
    return len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0

# Compute Jaccard Similarity between resumes and job descriptions
jaccard_scores = []
resume_ids, job_ids, job_labels = [], [], []

for _, resume in resumes.iterrows():
    for _, job in jobs.iterrows():
        score = compute_jaccard_similarity(resume['extracted_raw_text'], job['extracted_raw_text'])
        jaccard_scores.append(score)
        resume_ids.append(resume['resume_id'])
        job_ids.append(job['job_id'])
        job_labels.append(job['encoded_job_title'])

# Create DataFrame
X = pd.DataFrame({'resume_id': resume_ids, 'job_id': job_ids, 'jaccard_score': jaccard_scores})
y = np.array(job_labels)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X[['jaccard_score']], y, test_size=0.2, random_state=42)

# Train XGBoost model
model = xgb.XGBClassifier(objective='multi:softprob', eval_metric='mlogloss')
model.fit(X_train, y_train)

# Predict job titles for resumes
predictions = model.predict(X_test)
predicted_titles = label_encoder.inverse_transform(predictions)

# Determine suitability
def get_suitability(score):
    if score >= 0.75:
        return "Highly Suitable"
    elif score >= 0.50:
        return "Mildly Suitable"
    elif score >= 0.25:
        return "Less Suitable"
    else:
        return "Not Suitable"

X_test['predicted_job_title'] = predicted_titles
X_test['suitability'] = X_test['jaccard_score'].apply(get_suitability)



KeyboardInterrupt: 

In [ ]:
# Display results
for index, row in X_test.iterrows():
    print(f"-> Resume {row['resume_id']} Assessment:")
    print(f"    -> Predicted Relevant Job: \"{row['predicted_job_title']}\"")
    print(f"    -> Percentage: {row['jaccard_score'] * 100:.2f}%")
    print(f"    -> Status: \"{row['suitability']}\"\n")
